In [1]:
# ============================================================
# 15B_AURORA_probability_quality_and_stress_period_diagnostics.ipynb
#
# One-cell standalone version
#
# Purpose:
# 1. Generate S24 probability-quality diagnostics from Notebook 07 outputs.
# 2. Generate S24b reliability-bin diagnostics.
# 3. Generate S25 stress-period and market-condition decomposition from Notebook 13B outputs.
# 4. Do not retrain models.
# 5. Do not change source-aware return matrices, strategy selection, or main reported results.
#
# Educational/research use only.
# Not personalized financial advice.
# ============================================================

from __future__ import annotations

import json
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from sklearn.metrics import (
        f1_score,
        balanced_accuracy_score,
        cohen_kappa_score,
        mean_absolute_error,
    )
    HAS_SKLEARN = True
except Exception:
    HAS_SKLEARN = False
    print("sklearn not available. Some classification metrics will be set to NaN.")

# ============================================================
# 1. Paths and configuration
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

DATA_ROOT = PUBLICATION_ROOT / "data"
MODELING_DIR = DATA_ROOT / "modeling"
MODEL_DATA_PATH = MODELING_DIR / "AURORA_TWETF_features_with_labels.parquet"

AURORA_OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / "AURORA_TWETF"
COMPARISON_OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / "ROMA_AURORA_TWETF"

# Set to a string like "20260624_031817" if you want to force a specific run.
# If None, the notebook uses the latest available run directory.
NOTEBOOK07_RUN_ID = None
NOTEBOOK13B_RUN_ID = None

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

OUTPUT_ROOT = AURORA_OUTPUT_ROOT / "supplementary_probability_stress_diagnostics"
RUN_ROOT = OUTPUT_ROOT / f"run_{RUN_ID}"
TABLE_DIR = RUN_ROOT / "tables"
FIGURE_DIR = RUN_ROOT / "figures"
REPORT_DIR = RUN_ROOT / "reports"

for d in [OUTPUT_ROOT, RUN_ROOT, TABLE_DIR, FIGURE_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

ANNUALIZATION = 252
N_CLASSES = 5
CLASS_LABELS = list(range(N_CLASSES))

PRIMARY_AURORA_POLICY = "AURORA10_UAMV_B_more60_defensive"
PRIMARY_ROMA_POLICY = "ROMA_P4_balanced_regime_blend"
ROMA_B12 = "ROMA_B12_ma_timing_equal_weight"
ROMA_B6 = "ROMA_B6_00881_only"

DISPLAY_NAMES = {
    PRIMARY_AURORA_POLICY: "AURORA10-UAMV-B",
    PRIMARY_ROMA_POLICY: "ROMA-P4",
    ROMA_B12: "ROMA-B12",
    ROMA_B6: "ROMA-B6",
}

TARGET_COLS = [
    "TAIEX_regime_fixed_20d",
    "TAIEX_regime_fixed_60d",
]

TARGET_HORIZON_MAP = {
    "TAIEX_regime_fixed_20d": 20,
    "TAIEX_regime_fixed_60d": 60,
}

SELECTED_MODEL_NAME = "E1_validation_weighted_probability_ensemble"

print("=" * 90)
print("Notebook 15B: Probability-quality and stress-period diagnostics")
print("=" * 90)
print("Run timestamp UTC:", RUN_TIMESTAMP)
print("Run ID           :", RUN_ID)
print("Run root         :", RUN_ROOT)
print("=" * 90)

# ============================================================
# 2. Utility functions
# ============================================================

def latest_run(root: Path, prefix: str = "run_") -> Path:
    root = Path(root)
    runs = sorted([p for p in root.glob(f"{prefix}*") if p.is_dir()])
    if not runs:
        raise FileNotFoundError(f"No run directories found under {root}")
    return runs[-1]

def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path}")

def ensure_datetime_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"])
        out = out.set_index("date")
    out.index = pd.to_datetime(out.index)
    out = out.sort_index()
    out.index.name = "date"
    return out

def normalize_proba(p: np.ndarray) -> np.ndarray:
    p = np.asarray(p, dtype=float)
    p = np.nan_to_num(p, nan=0.0, posinf=0.0, neginf=0.0)
    p[p < 0] = 0.0

    row_sums = p.sum(axis=1, keepdims=True)
    zero_rows = row_sums[:, 0] <= 0

    if np.any(zero_rows):
        p[zero_rows, :] = 1.0 / p.shape[1]
        row_sums = p.sum(axis=1, keepdims=True)

    return p / row_sums

def proba_cols():
    return [f"proba_class_{i}" for i in range(N_CLASSES)]

def multiclass_brier_score(y_true, proba):
    y_true = np.asarray(y_true, dtype=int)
    proba = normalize_proba(proba)

    y_onehot = np.zeros_like(proba)
    valid = (y_true >= 0) & (y_true < proba.shape[1])
    y_onehot[np.arange(len(y_true))[valid], y_true[valid]] = 1.0

    return float(np.mean(np.sum((proba - y_onehot) ** 2, axis=1)))

def expected_calibration_error(y_true, proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    proba = normalize_proba(proba)

    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y_true).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf >= lo) & (conf < hi)

        if mask.sum() == 0:
            continue

        bin_acc = correct[mask].mean()
        bin_conf = conf[mask].mean()
        ece += mask.mean() * abs(bin_acc - bin_conf)

    return float(ece)

def probability_diagnostics(proba):
    proba = normalize_proba(proba)
    class_values = np.arange(proba.shape[1], dtype=float)

    expected_class = proba @ class_values

    entropy = -np.sum(
        np.clip(proba, 1e-12, 1.0) * np.log(np.clip(proba, 1e-12, 1.0)),
        axis=1,
    )
    normalized_entropy = entropy / np.log(proba.shape[1])

    bearish_probability = proba[:, 0] + proba[:, 1]
    bullish_probability = proba[:, 3] + proba[:, 4]
    max_probability = proba.max(axis=1)

    sorted_p = np.sort(proba, axis=1)
    probability_margin = sorted_p[:, -1] - sorted_p[:, -2]

    ordinal_variance = (proba @ (class_values ** 2)) - expected_class ** 2

    return {
        "expected_class": expected_class,
        "normalized_entropy": normalized_entropy,
        "bearish_probability": bearish_probability,
        "bullish_probability": bullish_probability,
        "max_probability": max_probability,
        "probability_margin": probability_margin,
        "ordinal_variance": ordinal_variance,
    }

def quadratic_weighted_kappa(y_true, y_pred):
    if not HAS_SKLEARN:
        return np.nan
    try:
        return float(cohen_kappa_score(y_true, y_pred, weights="quadratic", labels=CLASS_LABELS))
    except Exception:
        return np.nan

def classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    if not HAS_SKLEARN:
        return {
            "macro_f1": np.nan,
            "balanced_accuracy": np.nan,
            "quadratic_weighted_kappa": np.nan,
            "ordinal_mae": float(np.mean(np.abs(y_true - y_pred))),
        }

    return {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "quadratic_weighted_kappa": quadratic_weighted_kappa(y_true, y_pred),
        "ordinal_mae": float(mean_absolute_error(y_true, y_pred)),
    }

def reliability_bins(y_true, proba, n_bins=10):
    y_true = np.asarray(y_true, dtype=int)
    proba = normalize_proba(proba)

    conf = proba.max(axis=1)
    pred = proba.argmax(axis=1)
    correct = (pred == y_true).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    rows = []

    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf >= lo) & (conf < hi)

        rows.append({
            "bin": i + 1,
            "conf_low": lo,
            "conf_high": hi,
            "n_obs": int(mask.sum()),
            "mean_confidence": float(conf[mask].mean()) if mask.sum() else np.nan,
            "accuracy": float(correct[mask].mean()) if mask.sum() else np.nan,
        })

    return pd.DataFrame(rows)

def calculate_drawdown(equity):
    equity = pd.Series(equity).astype(float)
    running_max = equity.cummax()
    return equity / running_max - 1.0

def make_equity(r):
    r = pd.Series(r).dropna().astype(float)
    return (1.0 + r).cumprod()

def performance_metrics(r, annualization=ANNUALIZATION):
    r = pd.Series(r).dropna().astype(float)

    if len(r) == 0:
        return {
            "n_days": 0,
            "total_return": np.nan,
            "annual_return": np.nan,
            "annual_volatility": np.nan,
            "sharpe": np.nan,
            "sortino": np.nan,
            "max_drawdown": np.nan,
            "calmar": np.nan,
            "mean_daily_return": np.nan,
            "worst_daily_return": np.nan,
            "positive_day_rate": np.nan,
        }

    equity = make_equity(r)
    n = len(r)

    total_return = float(equity.iloc[-1] - 1.0)
    annual_return = float(equity.iloc[-1] ** (annualization / n) - 1.0)

    daily_vol = float(r.std(ddof=1)) if n > 1 else np.nan
    annual_vol = float(daily_vol * np.sqrt(annualization)) if np.isfinite(daily_vol) else np.nan

    mean_daily = float(r.mean())
    sharpe = float((mean_daily / daily_vol) * np.sqrt(annualization)) if daily_vol and daily_vol > 0 else np.nan

    downside = r[r < 0]
    downside_vol = float(downside.std(ddof=1)) if len(downside) > 1 else np.nan
    sortino = float((mean_daily / downside_vol) * np.sqrt(annualization)) if np.isfinite(downside_vol) and downside_vol > 0 else np.nan

    dd = calculate_drawdown(equity)
    max_dd = float(dd.min())
    calmar = float(annual_return / abs(max_dd)) if max_dd < 0 else np.nan

    return {
        "n_days": int(n),
        "total_return": total_return,
        "annual_return": annual_return,
        "annual_volatility": annual_vol,
        "sharpe": sharpe,
        "sortino": sortino,
        "max_drawdown": max_dd,
        "calmar": calmar,
        "mean_daily_return": mean_daily,
        "worst_daily_return": float(r.min()),
        "positive_day_rate": float((r > 0).mean()),
    }

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(stat.st_mtime, timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

# ============================================================
# 3. Locate Notebook 07 and Notebook 13B outputs
# ============================================================

NB07_ROOT_BASE = AURORA_OUTPUT_ROOT / "purged_walk_forward_models"
NB13B_ROOT_BASE = COMPARISON_OUTPUT_ROOT / "source_aware_unified_paper_comparison"

if NOTEBOOK07_RUN_ID is None:
    NB07_RUN_ROOT = latest_run(NB07_ROOT_BASE)
else:
    NB07_RUN_ROOT = NB07_ROOT_BASE / f"run_{NOTEBOOK07_RUN_ID}"

if NOTEBOOK13B_RUN_ID is None:
    NB13B_RUN_ROOT = latest_run(NB13B_ROOT_BASE)
else:
    NB13B_RUN_ROOT = NB13B_ROOT_BASE / f"run_{NOTEBOOK13B_RUN_ID}"

NB07_PROBA_PATH = NB07_RUN_ROOT / "probabilities" / "probabilities_all_purged_walk_forward_models.parquet"
NB07_PRED_PATH = NB07_RUN_ROOT / "predictions" / "predictions_all_purged_walk_forward_models.parquet"
NB07_SELECTED_PATH = NB07_RUN_ROOT / "selected_models_for_allocation_purged_walk_forward.csv"

NB13B_RETURN_MATRIX_PATH = NB13B_RUN_ROOT / "returns" / "notebook13B_source_aware_strict_test_return_matrix.parquet"

print("\n" + "=" * 90)
print("Input paths")
print("=" * 90)
print("Notebook 07 run root :", NB07_RUN_ROOT)
print("Notebook 13B run root:", NB13B_RUN_ROOT)
print("Probability file     :", NB07_PROBA_PATH)
print("Prediction file      :", NB07_PRED_PATH)
print("Selected models      :", NB07_SELECTED_PATH)
print("Return matrix        :", NB13B_RETURN_MATRIX_PATH)
print("Modeling dataset     :", MODEL_DATA_PATH)

for p in [NB07_PROBA_PATH, NB07_PRED_PATH, NB07_SELECTED_PATH, NB13B_RETURN_MATRIX_PATH, MODEL_DATA_PATH]:
    if not Path(p).exists():
        raise FileNotFoundError(p)

# ============================================================
# 4. Load inputs
# ============================================================

proba_all = ensure_datetime_index(read_table(NB07_PROBA_PATH))
pred_all = ensure_datetime_index(read_table(NB07_PRED_PATH))
selected_models = read_table(NB07_SELECTED_PATH)

strict_return_mat = ensure_datetime_index(read_table(NB13B_RETURN_MATRIX_PATH))
strict_dates = strict_return_mat.index

model_df = ensure_datetime_index(pd.read_parquet(MODEL_DATA_PATH))

print("\nLoaded shapes:")
print("Probability rows:", proba_all.shape)
print("Prediction rows :", pred_all.shape)
print("Selected models :", selected_models.shape)
print("Return matrix   :", strict_return_mat.shape)
print("Modeling data   :", model_df.shape)

# Validate probability columns.
missing_proba_cols = [c for c in proba_cols() if c not in proba_all.columns]
if missing_proba_cols:
    raise ValueError(f"Missing probability columns: {missing_proba_cols}")

# ============================================================
# 5. S24 probability-quality diagnostics
# ============================================================

print("\n" + "=" * 90)
print("Generating S24 probability-quality diagnostics")
print("=" * 90)

s24_rows = []

for target_col in TARGET_COLS:
    sel_rows = selected_models[selected_models["target_col"] == target_col]
    if sel_rows.empty:
        print("No selected model row for", target_col)
        continue

    selected_model = sel_rows.iloc[0]["selected_model"]
    horizon = TARGET_HORIZON_MAP[target_col]

    p_tmp = proba_all[
        (proba_all["target_col"] == target_col)
        & (proba_all["model_name"] == selected_model)
        & (proba_all["split"] == "test")
    ].copy()

    y_tmp = pred_all[
        (pred_all["target_col"] == target_col)
        & (pred_all["model_name"] == selected_model)
        & (pred_all["split"] == "test")
    ].copy()

    common = p_tmp.index.intersection(y_tmp.index).sort_values()
    p_tmp = p_tmp.loc[common]
    y_tmp = y_tmp.loc[common]

    common_strict = common.intersection(strict_dates).sort_values()

    for sample_name, idx in [
        ("pooled_test_folds", common),
        ("strict_test_overlap", common_strict),
    ]:
        if len(idx) == 0:
            print(f"No observations for {target_col} {sample_name}")
            continue

        p = normalize_proba(p_tmp.loc[idx, proba_cols()].values)
        y_true = y_tmp.loc[idx, "y_true"].astype(int).values
        y_pred = y_tmp.loc[idx, "y_pred"].astype(int).values

        diag = probability_diagnostics(p)
        cls = classification_metrics(y_true, y_pred)

        s24_rows.append({
            "target_col": target_col,
            "horizon": horizon,
            "sample": sample_name,
            "selected_model": selected_model,
            "n_obs": int(len(idx)),
            "date_start": str(pd.DatetimeIndex(idx).min().date()),
            "date_end": str(pd.DatetimeIndex(idx).max().date()),
            "macro_f1": cls["macro_f1"],
            "balanced_accuracy": cls["balanced_accuracy"],
            "quadratic_weighted_kappa": cls["quadratic_weighted_kappa"],
            "ordinal_mae": cls["ordinal_mae"],
            "ece_10bin": expected_calibration_error(y_true, p, n_bins=10),
            "brier_score": multiclass_brier_score(y_true, p),
            "mean_normalized_entropy": float(np.mean(diag["normalized_entropy"])),
            "median_normalized_entropy": float(np.median(diag["normalized_entropy"])),
            "mean_bearish_probability": float(np.mean(diag["bearish_probability"])),
            "median_bearish_probability": float(np.median(diag["bearish_probability"])),
            "mean_bullish_probability": float(np.mean(diag["bullish_probability"])),
            "mean_max_probability": float(np.mean(diag["max_probability"])),
            "mean_probability_margin": float(np.mean(diag["probability_margin"])),
            "mean_ordinal_variance": float(np.mean(diag["ordinal_variance"])),
        })

s24_df = pd.DataFrame(s24_rows)

# Add S13 validation-selected metrics for context if available.
rename_map = {
    "validation_mean_macro_f1": "s13_validation_macro_f1",
    "validation_mean_balanced_accuracy": "s13_validation_balanced_accuracy",
    "validation_mean_qwk": "s13_validation_quadratic_weighted_kappa",
    "validation_mean_ordinal_mae": "s13_validation_ordinal_mae",
    "validation_mean_ece": "s13_validation_ece_10bin",
}

selected_renamed = selected_models.rename(columns=rename_map).copy()
selected_cols = ["target_col", "selected_model"] + [
    c for c in rename_map.values() if c in selected_renamed.columns
]
selected_small = selected_renamed[selected_cols].copy()

if not s24_df.empty:
    s24_df = s24_df.merge(selected_small, on=["target_col", "selected_model"], how="left")

    for c in s24_df.columns:
        if pd.api.types.is_numeric_dtype(s24_df[c]):
            if c in ["horizon", "n_obs"]:
                continue
            s24_df[c] = s24_df[c].round(4)

s24_path = TABLE_DIR / "table_S24_probability_quality_diagnostics.csv"
s24_df.to_csv(s24_path, index=False)

print("S24 table:")
print(s24_df.to_string(index=False))
print("Saved:", s24_path)

# ============================================================
# 6. S24b reliability-bin diagnostics and figures
# ============================================================

print("\n" + "=" * 90)
print("Generating S24b reliability-bin diagnostics and figures")
print("=" * 90)

reliability_rows = []

for target_col in TARGET_COLS:
    sel_rows = selected_models[selected_models["target_col"] == target_col]
    if sel_rows.empty:
        continue

    selected_model = sel_rows.iloc[0]["selected_model"]

    p_tmp = proba_all[
        (proba_all["target_col"] == target_col)
        & (proba_all["model_name"] == selected_model)
        & (proba_all["split"] == "test")
    ].copy()

    y_tmp = pred_all[
        (pred_all["target_col"] == target_col)
        & (pred_all["model_name"] == selected_model)
        & (pred_all["split"] == "test")
    ].copy()

    common = p_tmp.index.intersection(y_tmp.index).sort_values()

    if len(common) == 0:
        continue

    p = normalize_proba(p_tmp.loc[common, proba_cols()].values)
    y = y_tmp.loc[common, "y_true"].astype(int).values

    rb = reliability_bins(y, p, n_bins=10)
    rb.insert(0, "target_col", target_col)
    rb.insert(1, "selected_model", selected_model)
    rb.insert(2, "sample", "pooled_test_folds")
    reliability_rows.append(rb)

    # Plot only non-empty bins.
    rb_plot = rb.dropna(subset=["mean_confidence", "accuracy"]).copy()

    plt.figure(figsize=(5.8, 5.8))
    plt.plot([0, 1], [0, 1], "--", color="gray", linewidth=1.2, label="Perfect calibration")
    if not rb_plot.empty:
        plt.plot(
            rb_plot["mean_confidence"],
            rb_plot["accuracy"],
            marker="o",
            linewidth=1.8,
            label="Observed",
        )
    plt.xlabel("Mean confidence")
    plt.ylabel("Accuracy")
    plt.title(f"Reliability: {target_col}")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    fig_path = FIGURE_DIR / f"figure_S24_reliability_{target_col}.png"
    plt.savefig(fig_path, dpi=240)
    plt.close()

    print("Saved reliability figure:", fig_path)

if reliability_rows:
    reliability_df = pd.concat(reliability_rows, ignore_index=True)
else:
    reliability_df = pd.DataFrame()

if not reliability_df.empty:
    for c in reliability_df.columns:
        if pd.api.types.is_numeric_dtype(reliability_df[c]):
            reliability_df[c] = reliability_df[c].round(4)

s24b_path = TABLE_DIR / "table_S24b_reliability_bins.csv"
reliability_df.to_csv(s24b_path, index=False)

print("S24b reliability bins:")
print(reliability_df.to_string(index=False))
print("Saved:", s24b_path)

# ============================================================
# 7. Construct market-condition labels for S25
# ============================================================

print("\n" + "=" * 90)
print("Constructing market-condition labels for S25")
print("=" * 90)

print("Available columns containing TAIEX:")
taiex_cols_preview = [c for c in model_df.columns if "TAIEX" in c.upper()]
print(taiex_cols_preview[:80])

candidate_return_cols = [
    c for c in model_df.columns
    if "TAIEX" in c.upper()
    and ("RET" in c.upper() or "RETURN" in c.upper())
    and model_df[c].dtype.kind in "fc"
]

candidate_close_cols = [
    c for c in model_df.columns
    if "TAIEX" in c.upper()
    and ("CLOSE" in c.upper() or "PX" in c.upper() or "PRICE" in c.upper())
    and model_df[c].dtype.kind in "fc"
]

market_proxy_used = None

if candidate_return_cols:
    # Prefer simple daily return-looking columns if multiple exist.
    preferred = [
        c for c in candidate_return_cols
        if any(x in c.upper() for x in ["RET_1D", "RETURN_1D", "DAILY"])
    ]
    market_ret_col = preferred[0] if preferred else candidate_return_cols[0]
    taiex_ret = model_df[market_ret_col].astype(float).copy()
    market_proxy_used = market_ret_col
    print("Using TAIEX return column:", market_ret_col)

elif candidate_close_cols:
    market_close_col = candidate_close_cols[0]
    taiex_ret = model_df[market_close_col].astype(float).pct_change()
    market_proxy_used = market_close_col + " pct_change"
    print("Using TAIEX close column:", market_close_col)

else:
    # Fallback: use equal-weight benchmark as market-condition proxy.
    fallback_col_candidates = [
        "AURORA_B1_equal_weight_all_etfs",
        "ROMA_B1_equal_weight_all_etfs",
        "AURORA_B6_00881_only",
        "ROMA_B6_00881_only",
    ]
    fallback_col = None
    for c in fallback_col_candidates:
        if c in strict_return_mat.columns:
            fallback_col = c
            break

    if fallback_col is None:
        raise ValueError(
            "Could not locate TAIEX return/close column or fallback market proxy in strict return matrix."
        )

    taiex_ret = strict_return_mat[fallback_col].copy()
    market_proxy_used = f"{fallback_col} fallback proxy"
    print("WARNING: No TAIEX column found. Using fallback market proxy:", fallback_col)

taiex_ret = pd.Series(taiex_ret).dropna()
taiex_ret.index = pd.to_datetime(taiex_ret.index)
taiex_ret = taiex_ret.sort_index()

common_market_dates = strict_return_mat.index.intersection(taiex_ret.index).sort_values()

if len(common_market_dates) == 0:
    raise ValueError("No common dates between market return proxy and strict return matrix.")

taiex_ret_strict = taiex_ret.loc[common_market_dates]
strict_mat_aligned = strict_return_mat.loc[common_market_dates].copy()

taiex_vol20 = taiex_ret.rolling(20).std()
taiex_vol20_strict = taiex_vol20.reindex(common_market_dates)
vol_median = taiex_vol20_strict.median(skipna=True)

condition_masks = {
    "Full strict-test period": pd.Series(True, index=common_market_dates),
    "Negative market days": taiex_ret_strict < 0,
    "Positive market days": taiex_ret_strict > 0,
    "High-volatility days": taiex_vol20_strict > vol_median,
    "Low-volatility days": taiex_vol20_strict <= vol_median,
}

print("Market proxy used:", market_proxy_used)
print("Common market dates:", len(common_market_dates), common_market_dates.min().date(), "to", common_market_dates.max().date())
print("20-day volatility median:", vol_median)
print("Condition counts:")
for k, v in condition_masks.items():
    print(f"  {k}: {int(pd.Series(v).fillna(False).sum())}")

# ============================================================
# 8. Define drawdown and recovery windows
# ============================================================

print("\n" + "=" * 90)
print("Defining main drawdown and recovery windows")
print("=" * 90)

if PRIMARY_ROMA_POLICY in strict_return_mat.columns:
    ref_returns = strict_return_mat[PRIMARY_ROMA_POLICY].dropna()
    ref_name = PRIMARY_ROMA_POLICY
elif PRIMARY_AURORA_POLICY in strict_return_mat.columns:
    ref_returns = strict_return_mat[PRIMARY_AURORA_POLICY].dropna()
    ref_name = PRIMARY_AURORA_POLICY
else:
    ref_returns = taiex_ret_strict.dropna()
    ref_name = market_proxy_used

ref_returns = ref_returns.loc[ref_returns.index.intersection(common_market_dates)].dropna()

if len(ref_returns) < 10:
    raise ValueError("Not enough reference returns to define drawdown window.")

ref_equity = make_equity(ref_returns)
ref_dd = calculate_drawdown(ref_equity)

trough_date = ref_dd.idxmin()
peak_date = ref_equity.loc[:trough_date].idxmax()

post_trough_equity = ref_equity.loc[trough_date:]
prior_peak_value = ref_equity.loc[peak_date]
recovered = post_trough_equity[post_trough_equity >= prior_peak_value]

if len(recovered) > 0:
    recovery_end_date = recovered.index[0]
else:
    recovery_end_date = ref_equity.index[-1]

drawdown_mask = (common_market_dates >= peak_date) & (common_market_dates <= trough_date)
recovery_mask = (common_market_dates > trough_date) & (common_market_dates <= recovery_end_date)

condition_masks["Main drawdown episode"] = pd.Series(drawdown_mask, index=common_market_dates)
condition_masks["Post-drawdown recovery"] = pd.Series(recovery_mask, index=common_market_dates)

print("Drawdown reference:", ref_name)
print("Peak date         :", peak_date.date())
print("Trough date       :", trough_date.date())
print("Recovery end date :", recovery_end_date.date())
print("Main drawdown n   :", int(drawdown_mask.sum()))
print("Recovery n        :", int(recovery_mask.sum()))

# ============================================================
# 9. S25 stress-period and market-condition decomposition
# ============================================================

print("\n" + "=" * 90)
print("Generating S25 stress-period and market-condition decomposition")
print("=" * 90)

stress_policies = [
    PRIMARY_AURORA_POLICY,
    PRIMARY_ROMA_POLICY,
    ROMA_B12,
    ROMA_B6,
]

stress_policies = [p for p in stress_policies if p in strict_mat_aligned.columns]

if not stress_policies:
    raise ValueError("None of the requested stress policies are present in the strict return matrix.")

s25_rows = []

for condition_name, mask in condition_masks.items():
    mask = pd.Series(mask, index=common_market_dates).fillna(False)
    dates = common_market_dates[mask.values]

    if len(dates) < 5:
        print("Skipping condition with too few dates:", condition_name, len(dates))
        continue

    for policy in stress_policies:
        r = strict_mat_aligned.loc[dates, policy].dropna()

        if len(r) < 5:
            continue

        m = performance_metrics(r)

        s25_rows.append({
            "condition": condition_name,
            "strategy": DISPLAY_NAMES.get(policy, policy),
            "policy_name": policy,
            "n_days": m["n_days"],
            "start_date": str(pd.DatetimeIndex(r.index).min().date()),
            "end_date": str(pd.DatetimeIndex(r.index).max().date()),
            "total_return": m["total_return"],
            "annual_return": m["annual_return"],
            "annual_volatility": m["annual_volatility"],
            "sharpe": m["sharpe"],
            "sortino": m["sortino"],
            "max_drawdown": m["max_drawdown"],
            "calmar": m["calmar"],
            "worst_daily_return": m["worst_daily_return"],
            "positive_day_rate": m["positive_day_rate"],
        })

s25_df = pd.DataFrame(s25_rows)

for c in [
    "total_return",
    "annual_return",
    "annual_volatility",
    "sharpe",
    "sortino",
    "max_drawdown",
    "calmar",
    "worst_daily_return",
    "positive_day_rate",
]:
    if c in s25_df.columns:
        s25_df[c] = s25_df[c].astype(float).round(4)

s25_path = TABLE_DIR / "table_S25_stress_period_decomposition.csv"
s25_df.to_csv(s25_path, index=False)

print("S25 table:")
print(s25_df.to_string(index=False))
print("Saved:", s25_path)

# ============================================================
# 10. Optional stress-period drawdown figure
# ============================================================

print("\n" + "=" * 90)
print("Generating optional stress-period drawdown figure")
print("=" * 90)

plot_policies = [p for p in stress_policies if p in strict_mat_aligned.columns]

plt.figure(figsize=(11, 6))
for policy in plot_policies:
    dd = calculate_drawdown(make_equity(strict_mat_aligned[policy].dropna()))
    plt.plot(dd.index, dd.values, linewidth=1.8, label=DISPLAY_NAMES.get(policy, policy))

plt.axvspan(peak_date, trough_date, color="red", alpha=0.10, label="Main drawdown episode")
plt.axvline(trough_date, color="red", linestyle="--", alpha=0.8, linewidth=1.1)
plt.title("Stress-period drawdown diagnostic")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()

stress_fig_path = FIGURE_DIR / "figure_S25_stress_period_drawdown_diagnostic.png"
plt.savefig(stress_fig_path, dpi=240)
plt.close()

print("Saved stress drawdown figure:", stress_fig_path)

# ============================================================
# 11. Output index, validation report, and manifest
# ============================================================

print("\n" + "=" * 90)
print("Saving output index, validation report, and manifest")
print("=" * 90)

output_rows = [
    {
        "supplement_table": "S24",
        "artifact": "probability_quality_diagnostics",
        "path": str(s24_path),
        "description": "Probability-quality diagnostics for selected regime-probability inputs from Notebook 07.",
    },
    {
        "supplement_table": "S24b",
        "artifact": "reliability_bins",
        "path": str(s24b_path),
        "description": "Reliability-bin diagnostics for selected regime-probability inputs from Notebook 07.",
    },
    {
        "supplement_table": "S25",
        "artifact": "stress_period_decomposition",
        "path": str(s25_path),
        "description": "Stress-period and market-condition decomposition for central strategies from Notebook 13B returns.",
    },
    {
        "supplement_table": "S25",
        "artifact": "stress_period_drawdown_figure",
        "path": str(stress_fig_path),
        "description": "Drawdown figure with main drawdown episode shading.",
    },
]

for fig in sorted(FIGURE_DIR.glob("figure_S24_reliability_*.png")):
    output_rows.append({
        "supplement_table": "S24",
        "artifact": "reliability_figure",
        "path": str(fig),
        "description": f"Reliability plot: {fig.name}",
    })

output_index = pd.DataFrame(output_rows)
output_index_path = TABLE_DIR / "notebook15B_output_index.csv"
output_index.to_csv(output_index_path, index=False)

validation_report = {
    "notebook": "15B_AURORA_probability_quality_and_stress_period_diagnostics.ipynb",
    "run_id": RUN_ID,
    "run_timestamp_utc": RUN_TIMESTAMP,
    "purpose": (
        "Generate supplementary probability-quality diagnostics and stress-period decomposition "
        "from existing Notebook 07 and Notebook 13B outputs without retraining or changing main results."
    ),
    "input_paths": {
        "notebook07_run_root": str(NB07_RUN_ROOT),
        "notebook13B_run_root": str(NB13B_RUN_ROOT),
        "notebook07_probabilities": str(NB07_PROBA_PATH),
        "notebook07_predictions": str(NB07_PRED_PATH),
        "notebook07_selected_models": str(NB07_SELECTED_PATH),
        "notebook13B_return_matrix": str(NB13B_RETURN_MATRIX_PATH),
        "modeling_dataset": str(MODEL_DATA_PATH),
    },
    "outputs": output_index.to_dict(orient="records"),
    "strict_test_period": {
        "n_days": int(len(strict_return_mat)),
        "start_date": str(strict_return_mat.index.min().date()),
        "end_date": str(strict_return_mat.index.max().date()),
    },
    "market_condition_proxy": {
        "proxy_used": market_proxy_used,
        "n_common_market_dates": int(len(common_market_dates)),
        "volatility_definition": "Trailing 20-day standard deviation of market proxy returns",
        "high_volatility_definition": "Trailing 20-day volatility above strict-window median",
    },
    "drawdown_window_definition": {
        "reference_series": ref_name,
        "peak_date": str(peak_date.date()),
        "trough_date": str(trough_date.date()),
        "recovery_end_date": str(recovery_end_date.date()),
        "main_drawdown_n_days": int(drawdown_mask.sum()),
        "recovery_n_days": int(recovery_mask.sum()),
    },
    "interpretation_boundary": (
        "Diagnostics are supplementary. They are not used for model selection, allocation selection, "
        "or replacement of the main source-aware performance values."
    ),
    "educational_note": (
        "This notebook is for reproducible financial-machine-learning research only. "
        "It does not provide personalized financial advice or performance guarantees."
    ),
}

validation_report_path = REPORT_DIR / "NOTEBOOK15B_validation_report.json"
save_json(validation_report_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)
manifest_path = REPORT_DIR / "NOTEBOOK15B_file_manifest_SHA256.csv"
manifest_df.to_csv(manifest_path, index=False)

print("Output index:")
print(output_index.to_string(index=False))

print("\n" + "=" * 90)
print("NOTEBOOK 15B COMPLETE")
print("=" * 90)
print("Run ID              :", RUN_ID)
print("Run root            :", RUN_ROOT)
print("S24 table           :", s24_path)
print("S24b table          :", s24b_path)
print("S25 table           :", s25_path)
print("Stress figure       :", stress_fig_path)
print("Output index        :", output_index_path)
print("Validation report   :", validation_report_path)
print("Manifest            :", manifest_path)
print("=" * 90)

print("\nSuggested S19 rows:")
print("S19-14 | Probability-quality diagnostics | Notebook 15B: supplementary probability-quality and stress-period diagnostics | table_S24_probability_quality_diagnostics.csv")
print("S19-15 | Reliability-bin diagnostics | Notebook 15B: supplementary probability-quality and stress-period diagnostics | table_S24b_reliability_bins.csv")
print("S19-16 | Stress-period decomposition | Notebook 15B: supplementary probability-quality and stress-period diagnostics | table_S25_stress_period_decomposition.csv")

print("\nSuggested manuscript sentence:")
print("Supplementary Table S24 reports additional probability-quality diagnostics for the selected regime-probability inputs, and Supplementary Table S25 reports stress-period and market-condition decomposition.")

Mounted at /content/drive
Notebook 15B: Probability-quality and stress-period diagnostics
Run timestamp UTC: 2026-07-12T09:49:04Z
Run ID           : 20260712_094904
Run root         : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/supplementary_probability_stress_diagnostics/run_20260712_094904

Input paths
Notebook 07 run root : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817
Notebook 13B run root: /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_AURORA_TWETF/source_aware_unified_paper_comparison/run_20260625_065916
Probability file     : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/probabilities/probabilities_all_purged_walk_forward_models.parquet
Prediction file      : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/purged_walk_forward_models/run_20260624_031817/predictions/predictions_all_purged_walk_forward_models.parquet
Selected models      : /con